# Task 4: Gold Layer Dimensional Modeling

This notebook builds Gold tables from Silver as per requirements:
- **fact_sales:** One row per order line
- **dim_customer:** One row per customer
- **dim_product:** One row per product
- **dim_supplier:** One row per supplier
- **dim_geography:** One row per nation, including its parent region
- **vw_annual_sales_kpis:** Annual revenue, orders, active customers, and average order value

### Parameterized catalog_name
Every table/view references `{{catalog_name}}` as the catalog (set at runtime). All data is sourced from silver tables in that catalog.

**Outcome:** All gold tables/views created. Final reconciliation report checks sales totals and row counts vs silver. Validates dimension key uniqueness.


In [0]:
-- Parameter widget: Catalog Name
-- Dropdown created for catalog_name; set via widget, not SET statement.
-- Use :catalog_name as a parameter in your SQL below.


In [0]:
-- Gold dimension: Customer
CREATE OR REPLACE TABLE IDENTIFIER(:catalog_name || '.gold.dim_customer') AS
SELECT
  c_custkey AS customer_id,
  c_name AS customer_name,
  c_address AS customer_address,
  c_nationkey AS nation_id,
  c_phone AS customer_phone,
  c_acctbal AS account_balance,
  c_mktsegment AS market_segment,
  c_comment AS customer_comment
FROM IDENTIFIER(:catalog_name || '.silver.customer');


In [0]:
-- Gold dimension: Product
CREATE OR REPLACE TABLE IDENTIFIER(:catalog_name || '.gold.dim_product') AS
SELECT
  p_partkey AS product_id,
  p_name AS product_name,
  p_mfgr AS manufacturer,
  p_brand AS brand,
  p_type AS product_type,
  p_size AS product_size,
  p_container AS container,
  p_retailprice AS retail_price,
  p_comment AS product_comment
FROM IDENTIFIER(:catalog_name || '.silver.part');


In [0]:
-- Gold dimension: Supplier
CREATE OR REPLACE TABLE IDENTIFIER(:catalog_name || '.gold.dim_supplier') AS
SELECT
  s_suppkey AS supplier_id,
  s_name AS supplier_name,
  s_address AS supplier_address,
  s_nationkey AS nation_id,
  s_phone AS supplier_phone,
  s_acctbal AS account_balance,
  s_comment AS supplier_comment
FROM IDENTIFIER(:catalog_name || '.silver.supplier');


In [0]:
-- Gold dimension: Geography (Nation + Region)
CREATE OR REPLACE TABLE IDENTIFIER(:catalog_name || '.gold.dim_geography') AS
SELECT
  n.n_nationkey AS nation_id,
  n.n_name AS nation_name,
  n.n_regionkey AS region_id,
  r.r_name AS region_name,
  n.n_comment AS nation_comment,
  r.r_comment AS region_comment
FROM IDENTIFIER(:catalog_name || '.silver.nation') n
JOIN IDENTIFIER(:catalog_name || '.silver.region') r ON n.n_regionkey = r.r_regionkey;


In [0]:
-- Gold fact table: Sales (Order Line)
CREATE OR REPLACE TABLE IDENTIFIER(:catalog_name || '.gold.fact_sales') AS
SELECT
  l.l_orderkey AS order_id,
  l.l_linenumber AS line_number,
  l.l_partkey AS product_id,
  l.l_suppkey AS supplier_id,
  o.o_custkey AS customer_id,
  o.o_orderstatus AS order_status,
  o.o_totalprice AS order_total,
  o.o_orderdate AS order_date,
  l.l_quantity AS quantity,
  l.l_extendedprice AS extended_price,
  l.l_discount AS discount,
  l.l_tax AS tax,
  l.l_returnflag AS return_flag,
  l.l_linestatus AS line_status,
  l.l_shipdate AS ship_date,
  l.l_commitdate AS commit_date,
  l.l_receiptdate AS receipt_date,
  l.l_shipinstruct AS ship_instruction,
  l.l_shipmode AS ship_mode,
  l.l_comment AS line_comment
FROM IDENTIFIER(:catalog_name || '.silver.lineitem') l
JOIN IDENTIFIER(:catalog_name || '.silver.orders') o ON l.l_orderkey = o.o_orderkey;


In [0]:
-- Gold KPI View: Annual Sales KPIs
CREATE OR REPLACE VIEW IDENTIFIER(:catalog_name || '.gold.vw_annual_sales_kpis') AS
SELECT
  YEAR(o.o_orderdate) AS order_year,
  SUM(l.l_extendedprice * (1 - l.l_discount)) AS net_revenue,
  COUNT(DISTINCT o.o_orderkey) AS total_orders,
  COUNT(DISTINCT o.o_custkey) AS active_customers,
  (SUM(l.l_extendedprice * (1 - l.l_discount)) / COUNT(DISTINCT o.o_orderkey)) AS average_order_value
FROM IDENTIFIER(:catalog_name || '.silver.lineitem') l
JOIN IDENTIFIER(:catalog_name || '.silver.orders') o
  ON l.l_orderkey = o.o_orderkey
GROUP BY YEAR(o.o_orderdate)
ORDER BY order_year;
